# Analisis Exploratorio de Datos y Preprocesamiento
## Impacto de Redes Sociales en Estudiantes

## 1. Configuracion e Importacion de Librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import warnings
import joblib
import os

sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)
warnings.filterwarnings('ignore')

print('Librerias importadas correctamente')

Librerias importadas correctamente


## 2. Carga de Datos

In [2]:
df = pd.read_csv('Social_media_impact_on_life.csv')
print(f'Dimensiones: {df.shape}')
print(f'\nPrimeras filas:')
print(df.head())

Dimensiones: (1705, 11)

Primeras filas:
   Student_ID  Age  Gender Academic_Level Country  Avg_Daily_Usage_Hours  \
0         232   21    Male  Undergraduate   Other                    4.0   
1         564   23  Female  Undergraduate   Other                    1.6   
2         788   22    Male       Graduate  Canada                    4.6   
3         686   18    Male  Undergraduate   Other                    7.0   
4         608   24  Female    High School   Other                    7.5   

  Most_Used_Platform Affects_Academic_Performance  Sleep_Hours_Per_Night  \
0           Facebook                           No                    6.7   
1           LinkedIn                           No                    8.6   
2          Instagram                           No                    6.7   
3           Snapchat                          Yes                    5.4   
4           Facebook                          Yes                    5.0   

   Mental_Health_Score Overall_Impact  
0    

In [3]:
print('=== INFORMACION DEL DATASET ===')
print(df.info())

=== INFORMACION DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1705 entries, 0 to 1704
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Student_ID                    1705 non-null   int64  
 1   Age                           1705 non-null   int64  
 2   Gender                        1705 non-null   object 
 3   Academic_Level                1705 non-null   object 
 4   Country                       1705 non-null   object 
 5   Avg_Daily_Usage_Hours         1705 non-null   float64
 6   Most_Used_Platform            1705 non-null   object 
 7   Affects_Academic_Performance  1705 non-null   object 
 8   Sleep_Hours_Per_Night         1705 non-null   float64
 9   Mental_Health_Score           1705 non-null   float64
 10  Overall_Impact                1705 non-null   object 
dtypes: float64(3), int64(2), object(6)
memory usage: 146.6+ KB
None


## 3. FASE 2 CRISP-DM - Comprension de los Datos (EDA)

### 3.1 Estructura General

In [4]:
print('=== ESTRUCTURA GENERAL ===')
print(f'Forma: {df.shape}')
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'\nColumnas Numericas: {numeric_cols}')
print(f'\nColumnas Categoricas: {categorical_cols}')

=== ESTRUCTURA GENERAL ===
Forma: (1705, 11)

Columnas Numericas: ['Student_ID', 'Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score']

Columnas Categoricas: ['Gender', 'Academic_Level', 'Country', 'Most_Used_Platform', 'Affects_Academic_Performance', 'Overall_Impact']


### 3.2 Estadisticas Descriptivas

In [5]:
print('=== NUMERICAS ===')
print(df[numeric_cols].describe().round(2))

=== NUMERICAS ===
       Student_ID      Age  Avg_Daily_Usage_Hours  Sleep_Hours_Per_Night  \
count     1705.00  1705.00                1705.00                1705.00   
mean       439.51    20.85                   5.10                   6.60   
std        267.06     1.76                   1.68                   1.21   
min          1.00    18.00                   1.50                   3.80   
25%        214.00    19.00                   3.80                   5.60   
50%        427.00    21.00                   5.10                   6.60   
75%        640.00    22.00                   6.30                   7.50   
max       1000.00    24.00                   8.50                   9.60   

       Mental_Health_Score  
count              1705.00  
mean                  6.22  
std                   1.28  
min                   4.00  
25%                   5.00  
50%                   6.00  
75%                   7.00  
max                   9.00  


In [6]:
print('=== CATEGORICAS ===')
print(df[categorical_cols].describe())

=== CATEGORICAS ===
       Gender Academic_Level Country Most_Used_Platform  \
count    1705           1705    1705               1705   
unique      2              3     111                 12   
top      Male  Undergraduate   Other          Instagram   
freq      878            721     667                389   

       Affects_Academic_Performance Overall_Impact  
count                          1705           1705  
unique                            2              3  
top                             Yes       Negative  
freq                           1011            939  


### 3.3 Valores Nulos y Duplicados

In [7]:
print('=== NULOS ===')
print(df.isnull().sum())
print(f'\nFilas duplicadas: {df.duplicated().sum()}')
print(f'Student_ID unicos: {df["Student_ID"].nunique()}/{len(df)}')

=== NULOS ===
Student_ID                      0
Age                             0
Gender                          0
Academic_Level                  0
Country                         0
Avg_Daily_Usage_Hours           0
Most_Used_Platform              0
Affects_Academic_Performance    0
Sleep_Hours_Per_Night           0
Mental_Health_Score             0
Overall_Impact                  0
dtype: int64

Filas duplicadas: 0
Student_ID unicos: 1000/1705


### 3.4 Variable Target

In [8]:
print('=== TARGET DISTRIBUTION ===')
counts = df['Overall_Impact'].value_counts()
pcts = df['Overall_Impact'].value_counts(normalize=True) * 100
print(counts)
print(f'\nPorcentajes:')
print(pcts.round(1))

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Set2', len(counts))
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='black')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h, f'{int(h)}\n({h/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')
ax.set_title('Distribucion del Target', fontsize=13, fontweight='bold')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

=== TARGET DISTRIBUTION ===
Overall_Impact
Negative    939
Positive    499
Neutral     267
Name: count, dtype: int64

Porcentajes:
Overall_Impact
Negative    55.1
Positive    29.3
Neutral     15.7
Name: proportion, dtype: float64


### 3.5 Distribuciones Numericas

In [9]:
numeric_features = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
for idx, col in enumerate(numeric_features):
    axes[idx].hist(df[col], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    axes[idx].set_title(f'Distribucion de {col}')
    axes[idx].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.6 Distribuciones Categoricas

In [10]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

ax = axes[0]
df['Gender'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
ax.set_title('Genero')

ax = axes[1]
df['Academic_Level'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
ax.set_title('Nivel Academico')

ax = axes[2]
df['Country'].value_counts().head(10).plot(kind='barh', ax=ax, color=sns.color_palette('Set2'))
ax.set_title('Top 10 Paises')

ax = axes[3]
df['Most_Used_Platform'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('Set2'))
ax.set_title('Plataforma')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax = axes[4]
df['Affects_Academic_Performance'].value_counts().plot(kind='bar', ax=ax, color=['red', 'green'])
ax.set_title('Afecta Rendimiento')

plt.tight_layout()
plt.show()

### 3.7 Analisis Bivariado

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cols = ['Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score']
for idx, col in enumerate(cols):
    sns.boxplot(data=df, x='Overall_Impact', y=col, ax=axes[idx], palette='Set2')
    axes[idx].set_title(f'{col} vs Target')
plt.tight_layout()
plt.show()

In [12]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()
cats = ['Gender', 'Academic_Level', 'Most_Used_Platform', 'Affects_Academic_Performance']
for idx, col in enumerate(cats):
    sns.countplot(data=df, x=col, hue='Overall_Impact', ax=axes[idx], palette='Set2')
    axes[idx].set_title(f'{col} por Target')
    plt.setp(axes[idx].xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.8 Correlaciones

In [13]:
df_corr = df.copy()
le_temp = LabelEncoder()
df_corr['Target_enc'] = le_temp.fit_transform(df['Overall_Impact'])
numeric_with_target = numeric_cols + ['Target_enc']
corr = df_corr[numeric_with_target].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Matriz de Correlacion')
plt.tight_layout()
plt.show()

print('\nCorrelaciones con Target:')
print(corr['Target_enc'].sort_values(ascending=False))


Correlaciones con Target:
Target_enc               1.000000
Mental_Health_Score      0.848988
Sleep_Hours_Per_Night    0.747814
Age                      0.090528
Student_ID               0.034817
Avg_Daily_Usage_Hours   -0.779452
Name: Target_enc, dtype: float64


### 3.9 Outliers (IQR)

In [14]:
print('=== OUTLIERS DETECCION ===')
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = len(df[(df[col] < lower) | (df[col] > upper)])
    pct = (outliers / len(df)) * 100
    print(f'{col}: {outliers} ({pct:.2f}%)')

=== OUTLIERS DETECCION ===
Student_ID: 0 (0.00%)
Age: 0 (0.00%)
Avg_Daily_Usage_Hours: 0 (0.00%)
Sleep_Hours_Per_Night: 0 (0.00%)
Mental_Health_Score: 0 (0.00%)


### 3.10 Hallazgos Clave

- 1,705 filas, 11 columnas
- 1,000 Student_IDs unicos (hay duplicados)
- Sin valores nulos
- Target desbalanceado: 55% Negative, 29% Positive, 16% Neutral
- 111 paises unicos - requiere reduccion de cardinalidad

## 4. FASE 3 CRISP-DM - Preparacion de Datos

### 4.1 Decisiones de Preprocesamiento

- Student_ID: Eliminar
- Country: Top 15 + Other
- Numericas: StandardScaler
- Ordinales: OrdinalEncoder (Academic_Level)
- Nominales: OneHotEncoder
- Target: LabelEncoder

### 4.2 Limpieza Inicial

In [15]:
df_prep = df.copy()
print('Estado inicial:', df_prep.shape)

df_prep = df_prep.drop('Student_ID', axis=1)
print('Tras eliminar Student_ID:', df_prep.shape)

top_n = 15
top_countries = df_prep['Country'].value_counts().head(top_n).index.tolist()
df_prep['Country'] = df_prep['Country'].apply(lambda x: x if x in top_countries else 'Other')
print(f'Paises reducidos a: {df_prep["Country"].nunique()}')

Estado inicial: (1705, 11)
Tras eliminar Student_ID: (1705, 10)
Paises reducidos a: 15


### 4.3 Encoding Target

In [16]:
le = LabelEncoder()
y = le.fit_transform(df_prep['Overall_Impact'])
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Mapeo:')
for k, v in mapping.items():
    print(f'  {k} -> {v}')

Mapeo:
  Negative -> 0
  Neutral -> 1
  Positive -> 2


### 4.4 Separacion Train/Test

In [17]:
X = df_prep.drop('Overall_Impact', axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

X_train: (1364, 9)
X_test: (341, 9)
y_train: (1364,)
y_test: (341,)


### 4.5 Pipeline ColumnTransformer

In [18]:
numeric_feat = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score']
ordinal_feat = ['Academic_Level']
nominal_feat = ['Gender', 'Most_Used_Platform', 'Affects_Academic_Performance', 'Country']

num_trans = Pipeline([('scaler', StandardScaler())])
ord_trans = Pipeline([('enc', OrdinalEncoder(
    categories=[['High School', 'Undergraduate', 'Graduate']],
    handle_unknown='use_encoded_value', unknown_value=-1
))])
nom_trans = Pipeline([('enc', OneHotEncoder(
    handle_unknown='ignore', sparse_output=False, drop='if_binary'
))])

preprocessor = ColumnTransformer([
    ('num', num_trans, numeric_feat),
    ('ord', ord_trans, ordinal_feat),
    ('nom', nom_trans, nominal_feat)
])

print('Pipeline creado')

Pipeline creado


In [19]:
print('Ajustando preprocessor...')
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f'X_train procesado: {X_train_proc.shape}')
print(f'X_test procesado: {X_test_proc.shape}')

Ajustando preprocessor...
X_train procesado: (1364, 34)
X_test procesado: (341, 34)


### 4.6 Resumen

In [20]:
print('='*70)
print('RESUMEN FINAL'.center(70))
print('='*70)
print(f'\nDataset original: {df.shape}')
print(f'X_train: {X_train_proc.shape}')
print(f'X_test: {X_test_proc.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

                            RESUMEN FINAL                             

Dataset original: (1705, 11)
X_train: (1364, 34)
X_test: (341, 34)
y_train: (1364,)
y_test: (341,)


### 4.7 Exportar

In [21]:
os.makedirs('preprocesados', exist_ok=True)

np.save('preprocesados/X_train.npy', X_train_proc)
np.save('preprocesados/X_test.npy', X_test_proc)
np.save('preprocesados/y_train.npy', y_train)
np.save('preprocesados/y_test.npy', y_test)
joblib.dump(preprocessor, 'preprocesados/pipeline.pkl')
joblib.dump(le, 'preprocesados/label_encoder.pkl')

print('Datos exportados exitosamente')

Datos exportados exitosamente


### 4.8 Comparacion: StandardScaler vs MinMaxScaler

En esta seccion comparamos ambos metodos de escalado para las variables numericas:
- **StandardScaler**: centra los datos en media=0, std=1. Robusto ante outliers moderados.
- **MinMaxScaler**: comprime los datos al rango [0,1]. Sensible a outliers.

Las variables categoricas (OrdinalEncoder, OneHotEncoder) permanecen iguales en ambos pipelines.

In [22]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report
import time

# Pipeline con StandardScaler (ya lo tenemos, pero lo recreamos para comparar limpiamente)
preprocessor_standard = ColumnTransformer([
    ('num', Pipeline([('scaler', StandardScaler())]), numeric_feat),
    ('ord', ord_trans, ordinal_feat),
    ('nom', nom_trans, nominal_feat)
])

# Pipeline con MinMaxScaler
preprocessor_minmax = ColumnTransformer([
    ('num', Pipeline([('scaler', MinMaxScaler())]), numeric_feat),
    ('ord', Pipeline([('enc', OrdinalEncoder(
        categories=[['High School', 'Undergraduate', 'Graduate']],
        handle_unknown='use_encoded_value', unknown_value=-1
    ))]), ordinal_feat),
    ('nom', Pipeline([('enc', OneHotEncoder(
        handle_unknown='ignore', sparse_output=False, drop='if_binary'
    ))]), nominal_feat)
])

# Transformar datos con ambos pipelines
X_train_std = preprocessor_standard.fit_transform(X_train)
X_test_std = preprocessor_standard.transform(X_test)

X_train_mm = preprocessor_minmax.fit_transform(X_train)
X_test_mm = preprocessor_minmax.transform(X_test)

print(f'StandardScaler - X_train shape: {X_train_std.shape}')
print(f'MinMaxScaler   - X_train shape: {X_train_mm.shape}')

StandardScaler - X_train shape: (1364, 34)
MinMaxScaler   - X_train shape: (1364, 34)


In [23]:
# Comparar distribuciones de las variables numericas con ambos escaladores
fig, axes = plt.subplots(4, 2, figsize=(14, 16))

for i, col in enumerate(numeric_feat):
    col_idx = i  # Las numericas estan en las primeras posiciones del ColumnTransformer
    
    # StandardScaler
    axes[i, 0].hist(X_train_std[:, col_idx], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
    axes[i, 0].set_title(f'{col} - StandardScaler', fontweight='bold')
    axes[i, 0].axvline(x=0, color='red', linestyle='--', alpha=0.5, label='media=0')
    axes[i, 0].legend()
    
    # MinMaxScaler
    axes[i, 1].hist(X_train_mm[:, col_idx], bins=30, alpha=0.7, color='coral', edgecolor='black')
    axes[i, 1].set_title(f'{col} - MinMaxScaler', fontweight='bold')
    axes[i, 1].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='centro=0.5')
    axes[i, 1].legend()

plt.suptitle('Comparacion de Distribuciones: StandardScaler vs MinMaxScaler', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [24]:
# Resumen estadistico de ambos escalados (solo variables numericas)
print('='*70)
print('ESTADISTICAS DE LAS VARIABLES NUMERICAS ESCALADAS')
print('='*70)

import pandas as pd

stats_std = pd.DataFrame(X_train_std[:, :4], columns=numeric_feat).describe().round(3)
stats_mm = pd.DataFrame(X_train_mm[:, :4], columns=numeric_feat).describe().round(3)

print('\n--- StandardScaler ---')
print(stats_std)
print('\n--- MinMaxScaler ---')
print(stats_mm)

ESTADISTICAS DE LAS VARIABLES NUMERICAS ESCALADAS

--- StandardScaler ---
            Age  Avg_Daily_Usage_Hours  Sleep_Hours_Per_Night  \
count  1364.000               1364.000               1364.000   
mean      0.000                 -0.000                  0.000   
std       1.000                  1.000                  1.000   
min      -1.637                 -2.151                 -2.302   
25%      -1.069                 -0.785                 -0.816   
50%       0.066                 -0.014                  0.009   
75%       0.633                  0.699                  0.751   
max       1.768                  2.005                  2.484   

       Mental_Health_Score  
count             1364.000  
mean                 0.000  
std                  1.000  
min                 -1.741  
25%                 -0.952  
50%                 -0.164  
75%                  0.624  
max                  2.201  

--- MinMaxScaler ---
            Age  Avg_Daily_Usage_Hours  Sleep_Hours_Per_N

In [25]:
# Comparacion con modelo baseline: Logistic Regression + Cross-Validation estratificado
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('='*70)
print('COMPARACION CON LOGISTIC REGRESSION (BASELINE)')
print('='*70)

# StandardScaler
t0 = time.time()
scores_std = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    X_train_std, y_train, cv=skf, scoring='f1_weighted'
)
t_std = time.time() - t0

# MinMaxScaler
t0 = time.time()
scores_mm = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    X_train_mm, y_train, cv=skf, scoring='f1_weighted'
)
t_mm = time.time() - t0

print(f'\nStandardScaler:')
print(f'  F1-weighted (CV): {scores_std.mean():.4f} (+/- {scores_std.std():.4f})')
print(f'  Tiempo: {t_std:.3f}s')

print(f'\nMinMaxScaler:')
print(f'  F1-weighted (CV): {scores_mm.mean():.4f} (+/- {scores_mm.std():.4f})')
print(f'  Tiempo: {t_mm:.3f}s')

# Diferencia
diff = abs(scores_std.mean() - scores_mm.mean())
mejor = 'StandardScaler' if scores_std.mean() > scores_mm.mean() else 'MinMaxScaler'
print(f'\nDiferencia: {diff:.4f} a favor de {mejor}')

COMPARACION CON LOGISTIC REGRESSION (BASELINE)

StandardScaler:
  F1-weighted (CV): 0.9423 (+/- 0.0159)
  Tiempo: 0.052s

MinMaxScaler:
  F1-weighted (CV): 0.9367 (+/- 0.0171)
  Tiempo: 0.056s

Diferencia: 0.0057 a favor de StandardScaler


In [26]:
# Visualizacion de la comparacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot de F1 scores
methods = ['StandardScaler', 'MinMaxScaler']
means = [scores_std.mean(), scores_mm.mean()]
stds = [scores_std.std(), scores_mm.std()]
colors = ['steelblue', 'coral']

bars = axes[0].bar(methods, means, yerr=stds, capsize=5, color=colors, edgecolor='black')
for bar, m in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
                f'{m:.4f}', ha='center', va='bottom', fontweight='bold')
axes[0].set_ylabel('F1-Score (weighted)')
axes[0].set_title('F1-Score por Metodo de Escalado (CV=5)', fontweight='bold')
axes[0].set_ylim(min(means) - 0.05, max(means) + 0.03)

# Boxplot de los folds
bp_data = [scores_std, scores_mm]
bp = axes[1].boxplot(bp_data, labels=methods, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('F1-Score (weighted)')
axes[1].set_title('Distribucion de Scores por Fold', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.9 Interpretacion de la Comparacion

**Observaciones:**
- **StandardScaler** centra cada variable en media=0 con desviacion estandar=1, preservando la forma de la distribucion original.
- **MinMaxScaler** comprime todos los valores al rango [0,1], lo cual puede ser preferible para algoritmos basados en distancias (KNN) o redes neuronales con activaciones sigmoid/tanh.
- En este dataset no hay outliers significativos, por lo que ambos metodos producen resultados similares.
- Se uso `class_weight='balanced'` en el modelo baseline para manejar el desbalanceo del target.

**Nota sobre el desbalanceo:** El target tiene 55% Negativo, 29% Positivo y 16% Neutro. Esto se abordara con mayor profundidad en la Fase 4 (Modelado), donde se pueden aplicar tecnicas como SMOTE, undersampling, o ajustes de class_weight segun el modelo.

## 5. FASE 4 CRISP-DM - Modelado

Entrenamos y comparamos 8 clasificadores usando **Stratified 10-Fold Cross-Validation**, siguiendo el patron del notebook de referencia.

**Clasificadores basicos:** KNN, Naive Bayes, Logistic Regression, Decision Tree, SVM

**Ensambles:** Random Forest, Gradient Boosting, AdaBoost

In [27]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                               AdaBoostClassifier)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone
from statistics import mean, stdev
import itertools

print('Clasificadores y utilidades importados')

Clasificadores y utilidades importados


### 5.1 Funcion de Cross-Validation para Multiples Clasificadores

Definimos la funcion `cvClassifiers` que entrena y evalua multiples clasificadores
usando Stratified K-Fold, acumulando predicciones de cada fold para construir
las matrices de confusion completas.

In [28]:
def cvClassifiers(X, Y, list_Classifiers, n_splits=10):
    """
    Entrena y evalua multiples clasificadores con Stratified K-Fold CV.
    
    Parametros:
    - X: features (array)
    - Y: target (array)
    - list_Classifiers: lista de instancias de clasificadores
    - n_splits: numero de folds (default=10)
    
    Retorna:
    - accuracy: lista de accuracy promedio (%) por clasificador
    - cm: lista de matrices de confusion por clasificador
    - all_scores: lista de scores por fold por clasificador (para analisis)
    """
    strtfdKFold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    kfold = strtfdKFold.split(X, Y)
    
    scores = [[] for c in list_Classifiers]
    predicted_y = [[] for c in list_Classifiers]
    real_y = [[] for c in list_Classifiers]
    
    for train_index, test_index in kfold:
        for i, c in enumerate(list_Classifiers):
            model = clone(c)  # Clonar para obtener instancia fresca
            X_train_cv, X_test_cv = X[train_index], X[test_index]
            y_train_cv, y_test_cv = Y[train_index], Y[test_index]
            
            model.fit(X_train_cv, y_train_cv)
            scores[i].append(model.score(X_test_cv, y_test_cv))
            
            py = model.predict(X_test_cv)
            predicted_y[i] = list(itertools.chain(predicted_y[i], py.flatten().tolist()))
            real_y[i] = list(itertools.chain(real_y[i], y_test_cv.flatten().tolist()))
    
    accuracy = [mean(sc) * 100 for sc in scores]
    cm = [confusion_matrix(real_y[i], predicted_y[i]) for i in range(len(list_Classifiers))]
    
    return accuracy, cm, scores

print('Funcion cvClassifiers definida')

Funcion cvClassifiers definida


### 5.2 Definicion y Entrenamiento de Clasificadores

In [29]:
# Definir los 8 clasificadores
nombres_clf = [
    'KNN (k=5)',
    'Naive Bayes',
    'Logistic Regression',
    'Decision Tree',
    'SVM (RBF)',
    'Random Forest',
    'Gradient Boosting',
    'AdaBoost'
]

list_Classifiers = [
    KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    GaussianNB(),
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    SVC(kernel='rbf', random_state=42, class_weight='balanced'),
    RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    AdaBoostClassifier(n_estimators=100, random_state=42)
]

# Ejecutar Cross-Validation con 10 folds
print('Ejecutando 10-Fold Stratified CV para 8 clasificadores...\n')

accuracies, confusion_matrices, all_scores = cvClassifiers(
    X_train_std, y_train, list_Classifiers, n_splits=10
)

# Mostrar resultados
print('='*60)
print('RESULTADOS - ACCURACY POR CLASIFICADOR (10-Fold CV)')
print('='*60)
for nombre, acc, sc in zip(nombres_clf, accuracies, all_scores):
    print(f'  {nombre:<25} {acc:.2f}% (+/- {stdev(sc)*100:.2f}%)')

print(f'\n  Mejor modelo: {nombres_clf[accuracies.index(max(accuracies))]} ({max(accuracies):.2f}%)')

Ejecutando 10-Fold Stratified CV para 8 clasificadores...



RESULTADOS - ACCURACY POR CLASIFICADOR (10-Fold CV)
  KNN (k=5)                 94.72% (+/- 2.85%)
  Naive Bayes               29.84% (+/- 3.75%)
  Logistic Regression       94.50% (+/- 2.03%)
  Decision Tree             97.21% (+/- 1.03%)
  SVM (RBF)                 96.63% (+/- 1.60%)
  Random Forest             98.83% (+/- 0.71%)
  Gradient Boosting         98.02% (+/- 1.25%)
  AdaBoost                  85.41% (+/- 4.10%)

  Mejor modelo: Random Forest (98.83%)


### 5.3 Matrices de Confusion (Cross-Validation)

Cada matriz de confusion se construye acumulando las predicciones de los 10 folds,
dando una vision completa del comportamiento del clasificador.

In [30]:
# Mostrar matrices de confusion para todos los clasificadores
target_names = ['Negative', 'Neutral', 'Positive']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i in range(len(list_Classifiers)):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrices[i], 
        display_labels=target_names
    )
    disp.plot(ax=axes[i], cmap=plt.cm.Blues, values_format='d')
    axes[i].set_title(f'{nombres_clf[i]}\nAcc: {accuracies[i]:.2f}%', fontweight='bold', fontsize=10)

plt.suptitle('Matrices de Confusion - 10-Fold Stratified CV', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.4 Comparacion Visual de Clasificadores

In [31]:
# Grafico de barras comparativo
fig, ax = plt.subplots(figsize=(12, 6))

colores = sns.color_palette('Set2', len(nombres_clf))
bars = ax.barh(nombres_clf, accuracies, color=colores, edgecolor='black')

for bar, acc in zip(bars, accuracies):
    ax.text(acc + 0.3, bar.get_y() + bar.get_height()/2, 
            f'{acc:.2f}%', va='center', fontweight='bold')

ax.set_xlabel('Accuracy (%)', fontsize=11)
ax.set_title('Comparacion de Accuracy - 10-Fold Stratified CV', fontsize=13, fontweight='bold')
ax.set_xlim(min(accuracies) - 5, max(accuracies) + 5)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [32]:
# Boxplot de accuracy por fold para cada clasificador
fig, ax = plt.subplots(figsize=(14, 6))

scores_pct = [[s * 100 for s in sc] for sc in all_scores]
bp = ax.boxplot(scores_pct, labels=nombres_clf, patch_artist=True, vert=True)

colores = sns.color_palette('Set2', len(nombres_clf))
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Distribucion de Accuracy por Fold (10-Fold CV)', fontsize=13, fontweight='bold')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.5 Metricas Detalladas del Mejor Modelo

Evaluamos el mejor clasificador en el **test set** (datos nunca vistos durante el CV).

In [33]:
# Identificar y entrenar el mejor modelo
idx_mejor = accuracies.index(max(accuracies))
mejor_nombre = nombres_clf[idx_mejor]
mejor_clf = clone(list_Classifiers[idx_mejor])

print(f'Mejor modelo: {mejor_nombre} (Accuracy CV: {accuracies[idx_mejor]:.2f}%)\n')

# Entrenar en todo el training set
mejor_clf.fit(X_train_std, y_train)
y_pred = mejor_clf.predict(X_test_std)

# Classification report completo
print('=== CLASSIFICATION REPORT (TEST SET) ===\n')
print(classification_report(y_test, y_pred, target_names=target_names))

# Matriz de confusion en test
fig, ax = plt.subplots(figsize=(8, 6))
cm_test = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=target_names)
disp.plot(ax=ax, cmap=plt.cm.Blues, values_format='d')
ax.set_title(f'Matriz de Confusion - {mejor_nombre} (Test Set)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Accuracy en test: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'F1-Score en test (weighted): {f1_score(y_test, y_pred, average="weighted"):.4f}')

Mejor modelo: Random Forest (Accuracy CV: 98.83%)

=== CLASSIFICATION REPORT (TEST SET) ===

              precision    recall  f1-score   support

    Negative       0.99      1.00      1.00       188
     Neutral       1.00      0.96      0.98        53
    Positive       0.99      1.00      1.00       100

    accuracy                           0.99       341
   macro avg       0.99      0.99      0.99       341
weighted avg       0.99      0.99      0.99       341

Accuracy en test: 99.41%
F1-Score en test (weighted): 0.9941


### 5.6 Optimizacion de Hiperparametros (GridSearchCV)

Aplicamos GridSearchCV al mejor modelo para buscar la combinacion optima de hiperparametros con 10-Fold CV.

In [34]:
# Grids de hiperparametros para cada modelo
param_grids = {
    'KNN (k=5)': {
        'n_neighbors': [3, 5, 7, 9, 11, 15],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    'Naive Bayes': {
        'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
    },
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['lbfgs', 'liblinear'],
        'class_weight': ['balanced']
    },
    'Decision Tree': {
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': ['balanced']
    },
    'SVM (RBF)': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto'],
        'class_weight': ['balanced']
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5],
        'class_weight': ['balanced']
    },
    'Gradient Boosting': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7]
    },
    'AdaBoost': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.5, 1.0]
    }
}

print(f'Optimizando: {mejor_nombre}')
print(f'Parametros a probar: {param_grids[mejor_nombre]}\n')

# GridSearchCV con 10-Fold
grid_search = GridSearchCV(
    estimator=clone(list_Classifiers[idx_mejor]),
    param_grid=param_grids[mejor_nombre],
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_std, y_train)

print(f'\nMejores parametros: {grid_search.best_params_}')
print(f'Mejor Accuracy (CV): {grid_search.best_score_*100:.2f}%')
print(f'Accuracy antes del tuning (CV): {accuracies[idx_mejor]:.2f}%')
print(f'Mejora: {(grid_search.best_score_*100 - accuracies[idx_mejor]):+.2f}%')

Optimizando: Random Forest
Parametros a probar: {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15, None], 'min_samples_split': [2, 5], 'class_weight': ['balanced']}

Fitting 10 folds for each of 24 candidates, totalling 240 fits



Mejores parametros: {'class_weight': 'balanced', 'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 100}
Mejor Accuracy (CV): 98.83%
Accuracy antes del tuning (CV): 98.83%
Mejora: -0.00%


In [35]:
# Evaluar modelo optimizado en test set
y_pred_tuned = grid_search.best_estimator_.predict(X_test_std)

print('=== CLASSIFICATION REPORT - MODELO OPTIMIZADO (TEST SET) ===\n')
print(classification_report(y_test, y_pred_tuned, target_names=target_names))

# Comparar matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

disp1 = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=target_names)
disp1.plot(ax=axes[0], cmap=plt.cm.Blues, values_format='d')
axes[0].set_title(f'{mejor_nombre}\nAntes del Tuning', fontweight='bold')

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_tuned, display_labels=target_names)
disp2.plot(ax=axes[1], cmap=plt.cm.Greens, values_format='d')
axes[1].set_title(f'{mejor_nombre}\nDespues del Tuning', fontweight='bold')

plt.tight_layout()
plt.show()

# Comparacion numerica
acc_before = accuracy_score(y_test, y_pred) * 100
acc_after = accuracy_score(y_test, y_pred_tuned) * 100
f1_before = f1_score(y_test, y_pred, average='weighted')
f1_after = f1_score(y_test, y_pred_tuned, average='weighted')

print(f'\n{"Metrica":<20} {"Antes":>10} {"Despues":>10} {"Diferencia":>12}')
print('-'*55)
print(f'{"Accuracy":<20} {acc_before:>9.2f}% {acc_after:>9.2f}% {acc_after-acc_before:>+11.2f}%')
print(f'{"F1-Score":<20} {f1_before:>10.4f} {f1_after:>10.4f} {f1_after-f1_before:>+12.4f}')

=== CLASSIFICATION REPORT - MODELO OPTIMIZADO (TEST SET) ===

              precision    recall  f1-score   support

    Negative       0.99      1.00      1.00       188
     Neutral       1.00      0.96      0.98        53
    Positive       0.99      1.00      1.00       100

    accuracy                           0.99       341
   macro avg       0.99      0.99      0.99       341
weighted avg       0.99      0.99      0.99       341


Metrica                   Antes    Despues   Diferencia
-------------------------------------------------------
Accuracy                 99.41%     99.41%       +0.00%
F1-Score                 0.9941     0.9941      +0.0000


### 5.7 Exportar Modelo Final

In [36]:
# Guardar modelo optimizado y artefactos
os.makedirs('modelo_final', exist_ok=True)

joblib.dump(grid_search.best_estimator_, 'modelo_final/mejor_modelo.pkl')
joblib.dump(preprocessor_standard, 'modelo_final/preprocessor.pkl')
joblib.dump(le, 'modelo_final/label_encoder.pkl')

# Resumen de todos los resultados
resumen = {
    'mejor_modelo': mejor_nombre,
    'mejores_parametros': grid_search.best_params_,
    'accuracy_cv': grid_search.best_score_ * 100,
    'accuracy_test': acc_after,
    'f1_test': f1_after,
    'todos_los_modelos': {n: {'accuracy': a, 'std': stdev(s)*100} 
                           for n, a, s in zip(nombres_clf, accuracies, all_scores)}
}
joblib.dump(resumen, 'modelo_final/resumen_resultados.pkl')

print('Modelo final exportado en carpeta modelo_final/')
print(f'  - mejor_modelo.pkl ({mejor_nombre})')
print(f'  - preprocessor.pkl')
print(f'  - label_encoder.pkl')
print(f'  - resumen_resultados.pkl')
print(f'\nMejores parametros: {grid_search.best_params_}')
print(f'Accuracy final (test): {acc_after:.2f}%')

Modelo final exportado en carpeta modelo_final/
  - mejor_modelo.pkl (Random Forest)
  - preprocessor.pkl
  - label_encoder.pkl
  - resumen_resultados.pkl

Mejores parametros: {'class_weight': 'balanced', 'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 100}
Accuracy final (test): 99.41%


## Conclusion

FASES 2, 3 y 4 del CRISP-DM completadas:

**FASE 2: Comprension de Datos (EDA)**
- Analisis estructural, estadistico
- Nulos, duplicados, outliers
- Univariado, bivariado, correlaciones

**FASE 3: Preparacion de Datos**
- Limpieza y transformacion
- Pipeline reproducible con ColumnTransformer
- Comparacion StandardScaler vs MinMaxScaler
- Train/test estratificado

**FASE 4: Modelado**
- Funcion cvClassifiers con 10-Fold Stratified CV
- Comparacion de 8 clasificadores (5 basicos + 3 ensambles)
- Matrices de confusion acumuladas por fold
- Optimizacion con GridSearchCV
- Modelo final exportado

Listo para FASE 5: Evaluacion detallada y FASE 6: Despliegue